# STEP 5 — RQ3: Cost-sensitive threshold + Profit/ROI + EMP

Operational model: **RAW/natural-distribution LightGBM** (Step 2 best_params), NO
calibration layer. Probabilities are **5-fold out-of-fold** (the model does not score a
customer it has seen — no leakage). For cost/value there is **no magic number**: parametric
(c, γ, per-customer CLV) + sensitivity. 5 sets independent; holdout does not enter. Heavy
logic in `src/profit.py`.

Profit matrix: TP=γ·CLV−c, FP=−c, FN=−CLV, TN=0 (standard cost-sensitive churn).
A = accuracy-threshold (t=0.5); B = profit-maximizing threshold t*. EMP = cross-set comparison
(per-customer expected maximum profit over a γ~Beta distribution, normalized to mean CLV).

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd


def _bul_kok():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "config.yaml").exists():
            return c
    raise RuntimeError("config.yaml not found")


KOK = _bul_kok()
if str(KOK) not in sys.path:
    sys.path.insert(0, str(KOK))

warnings.filterwarnings("ignore")
from src import config as cfg
from src import plotstyle as ps
from src import profit as pr
from src import strings as S

np.random.seed(cfg.SEED)
ps.uygula()
cfg.klasorleri_hazirla()
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)

CIKTI = []


def yaz(s=""):
    print(s)
    CIKTI.append(str(s))

## 1. CLV bases (defined in config; no magic number, all logged)

In [2]:
veriler = {k: pd.read_csv(cfg.PROCESSED / f"{k}_clean.csv") for k in cfg.DATASETS}
a, b = pr.PARAM["emp_beta"]
yaz(S.MSG5["emp_varsayim"].format(a=a, b=b, e=a / (a + b), c=int(pr.PARAM["emp_c_oran"] * 100)))
yaz(f"Parameters: horizon={pr.PARAM['ufuk_ay']} months | c ratios={pr.PARAM['c_oranlari']} | γ={pr.PARAM['gamma_listesi']}")

EMP assumption: γ ~ Beta(6,14) (E[γ]=0.30), reference cost = 5% of mean CLV.
Parameters: horizon=24 months | c ratios=[0.01, 0.02, 0.05, 0.1, 0.2] | γ=[0.2, 0.3, 0.4, 0.5]


## 2. Each set: OOF probability + (c,γ) sweep + profit-threshold + ROI + EMP
**(Longest step: one 5-fold CV per set.)**

In [3]:
yaz(S.MSG["bolum"].format(ad="PROFIT / ROI / EMP COMPUTATION"))
tum = {}
for k in cfg.DATASETS:
    t = time.time()
    tum[k] = pr.calistir_set(k, veriler[k], cfg.SEED)
    r = tum[k]
    yaz(S.MSG5["clv"].format(set=k, temel=r["temel"], ort=r["ort_clv"], medyan=r["medyan_clv"], sifir=r["sifir"]))
    yaz(f"  EMP(per customer, mean-CLV unit)={r['emp']:.4f}  ({time.time()-t:.0f}s)")

===== PROFIT / ROI / EMP COMPUTATION =====


telco: CLV basis = MonthlyCharges × 24 mo; mean=1554.3 median=1688.4 (zero CLV: 0)
  EMP(per customer, mean-CLV unit)=0.0491  (4s)


cell2cell: CLV basis = MonthlyRevenue × 24 mo; mean=1411.3 median=1163.0 (zero CLV: 9)
  EMP(per customer, mean-CLV unit)=0.0352  (18s)


ecommerce: CLV basis = CashbackAmount (monthly value proxy) × 24 mo; mean=4241.0 median=3920.2 (zero CLV: 3)
  EMP(per customer, mean-CLV unit)=0.0162  (14s)


iranian: CLV basis = Customer Value (direct); mean=471.0 median=228.5 (zero CLV: 132)
  EMP(per customer, mean-CLV unit)=0.0007  (16s)


bank: CLV basis = Balance × 0.02 annual margin × 2.0 yr; mean=3059.4 median=3887.9 (zero CLV: 3617)
  EMP(per customer, mean-CLV unit)=0.0319  (4s)


## 3. Tables
`rq3_profit_summary.csv` (set × c × γ: threshold A/B, profit A/B, ROI A/B, ROI increase, EMP)
and `rq3_clv_basis.csv`.

In [4]:
ozet = pr.tablo_ozet(tum)
clv_t = pr.tablo_clv(tum)
yaz(S.MSG["bolum"].format(ad="CLV BASES"))
yaz(clv_t.to_string(index=False))
yaz(S.MSG["bolum"].format(ad="PROFIT/ROI SUMMARY (all c,γ)"))
yaz(ozet.to_string(index=False))
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "rq3_profit_summary.csv"))
yaz(S.MSG["kayit"].format(yol=cfg.TABLES / "rq3_clv_basis.csv"))

===== CLV BASES =====
  Dataset                                    CLV basis  Mean CLV  Median CLV  Zero-CLV count
    telco                       MonthlyCharges × 24 mo   1554.28     1688.40               0
cell2cell                       MonthlyRevenue × 24 mo   1411.27     1163.04               9
ecommerce CashbackAmount (monthly value proxy) × 24 mo   4240.98     3920.16               3
  iranian                      Customer Value (direct)    470.97      228.48             132
     bank        Balance × 0.02 annual margin × 2.0 yr   3059.44     3887.94            3617
===== PROFIT/ROI SUMMARY (all c,γ) =====
  Dataset  Cost fraction   γ  Threshold A (0.5)  Threshold B (t*)    Profit A   Profit B    ROI A  ROI B  ROI gain (B−A)  Profit gain %  EMP (per customer)
    telco           0.01 0.2                0.5              0.01  -1144102.8   558671.0  -50.695  5.118          55.813          148.8              0.0491
    telco           0.01 0.3                0.5              0.01  

## 4. Figures
`outputs/figures/<set>/`: rq3_profit_vs_threshold, rq3_roi_sensitivity, rq3_strategy_comparison.
`outputs/figures/_rq3/`: rq3_emp_by_dataset.

In [5]:
yaz(S.MSG["bolum"].format(ad="FIGURES"))
for k in cfg.DATASETS:
    yollar = pr.figurler_set(k, tum[k])
    yaz(f"{k}: {len(yollar)} figures -> {cfg.FIGURES / k}")
yol_emp = pr.figur_emp(tum)
yaz(S.MSG["kayit"].format(yol=yol_emp))

===== FIGURES =====


telco: 3 figures -> /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/telco


cell2cell: 3 figures -> /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/cell2cell


ecommerce: 3 figures -> /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/ecommerce


iranian: 3 figures -> /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/iranian


bank: 3 figures -> /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/bank
Saved: /Users/emrahfidan/Desktop/churn-xai-profit/outputs/figures/_rq3/rq3_emp_by_dataset.png


## 5. Summary — RQ3 headline (decision left to the user)
The ROI/profit gain of the profit-threshold (B) over the accuracy-threshold (A), the EMP
ranking, parameter sensitivity (does the sign flip direction).

In [6]:
yaz(S.MSG["bolum"].format(ad="SUMMARY — RQ3"))
K = S.KOLON5
for k in cfg.DATASETS:
    sub = ozet[ozet[K["veri_seti"]] == k]
    ra = sub[K["roi_artis"]]
    kp = sub[K["kar_artis_yuzde"]]
    yaz(S.MSG5["set_ozet"].format(
        set=k,
        dusuk=f"ROI+{ra.min():.2f}/profit+{kp.min():.0f}%",
        yuksek=f"ROI+{ra.max():.2f}/profit+{kp.max():.0f}%",
        emp=tum[k]["emp"]))
    neg = int((sub[K["kar_b"]] < sub[K["kar_a"]]).sum())
    yaz(f"    number of (c,γ) where B<A: {neg}/{len(sub)} (if 0, the profit-threshold is at least as good as A in every scenario)")

emp_sirali = sorted(cfg.DATASETS, key=lambda s: tum[s]["emp"], reverse=True)
yaz("")
yaz(S.MSG5["emp_sira"].format(sira=" > ".join(f"{s}({tum[s]['emp']:.3f})" for s in emp_sirali)))
en_kazandiran = max(cfg.DATASETS, key=lambda s: ozet[ozet[K["veri_seti"]] == s][K["kar_artis_yuzde"]].median())
yaz(f"Set where the profit-threshold gains the most (median profit increase %): {en_kazandiran}")
yaz("")
yaz(S.MSG5["bitti"])

_log = cfg.LOGS / "adim5_ozet.log"
_log.write_text("\n".join(CIKTI) + "\n", encoding="utf-8")
print(S.MSG["kayit"].format(yol=_log))

===== SUMMARY — RQ3 =====
telco: profit-threshold ROI-gain range (c,γ sweep): ROI+2.33/profit+47% … ROI+55.81/profit+365%; EMP=0.0491
    number of (c,γ) where B<A: 0/20 (if 0, the profit-threshold is at least as good as A in every scenario)
cell2cell: profit-threshold ROI-gain range (c,γ sweep): ROI+18.97/profit+48% … ROI+386.78/profit+156%; EMP=0.0352
    number of (c,γ) where B<A: 0/20 (if 0, the profit-threshold is at least as good as A in every scenario)
ecommerce: profit-threshold ROI-gain range (c,γ sweep): ROI+-12.73/profit+15% … ROI+6.15/profit+4542%; EMP=0.0162
    number of (c,γ) where B<A: 0/20 (if 0, the profit-threshold is at least as good as A in every scenario)
iranian: profit-threshold ROI-gain range (c,γ sweep): ROI+-0.91/profit+5% … ROI+1.62/profit+1145%; EMP=0.0007
    number of (c,γ) where B<A: 0/20 (if 0, the profit-threshold is at least as good as A in every scenario)
bank: profit-threshold ROI-gain range (c,γ sweep): ROI+3.52/profit+42% … ROI+85.58/profit+261%; 